# V2 · Diagnóstico de calidad del valor contractual

Exploración previa a una posible versión **alertas-cd-2.1**. Este notebook analiza los resultados existentes de **2.0**, sin recalcular D1/D2/D3 ni modificar la clasificación.

Ejecute todas las celdas con el entorno `secop`. Las tablas y conclusiones aparecen aquí; no se exportan resultados. Se utiliza DuckDB con **1 GB y 1 hilo**, solo sobre columnas necesarias; los archivos temporales se eliminan al cerrar la consulta.

**Objetivo:** separar valores no utilizables para el análisis de señales de riesgo. Una marca estadística no demuestra un error de digitación ni permite adivinar el valor real.


In [ ]:
from pathlib import Path
import json, math, tempfile
import pandas as pd
import duckdb
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p/'secop_pipeline').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Abra V2 desde la carpeta del proyecto o notebooks.')
LOTE = 'desde_gold_v2_20261009'
TOPE_ABSOLUTO = 1e13  # 10 billones COP; criterio estricto: valor > tope.
FACTOR_MEDIANA = 1000.0  # Valor > factor × mediana del tipo y año.
display(pd.DataFrame([{'lote': LOTE, 'tope_COP': TOPE_ABSOLUTO, 'factor_mediana': FACTOR_MEDIANA}]))


## 1. Método del diagnóstico

- **A:** valor finito superior al tope absoluto.
- **B:** valor positivo y finito superior al factor por la mediana de su `tipo_normalizado` y año de firma.
- **C:** nulo, no finito o menor o igual a cero.

La mediana usa todos los valores finitos y positivos del tipo-año. También se calcula una mediana alternativa excluyendo primero A, para saber si cambia el conjunto marcado. Los grupos sin tipo o año no reciben una evaluación relativa; se cuentan por separado.

Los motivos son excluyentes: A y B, solo A, solo B, C. La columna `alerta_y_d2` cuenta la intersección de clasificación ALERTA y D2=1 actuales.


In [ ]:
def diagnosticar_valores(root, lote, tope, factor):
    """Lee el Parquet completo por columnas; devuelve solo tablas agregadas y muestras.

    No cambia datos, indicadores, reportes ni checkpoints del flujo.
    """
    if not math.isfinite(tope) or tope <= 0 or not math.isfinite(factor) or factor <= 0:
        raise ValueError('Tope y factor deben ser finitos y positivos.')
    tablas = {}
    folder=root/'salida/alertas'/('lote_'+lote)
    reports=list(folder.glob('*/ejecucion.json'))
    if len(reports)!=1:
        raise RuntimeError(f'Se esperaba un reporte inequívoco, encontrados {len(reports)}')
    manifest=reports[0]
    report=json.loads(manifest.read_text(encoding='utf-8'))
    if report['estado']!='COMPLETADA' or report['parametros']['version']!='alertas-cd-2.0':
        raise RuntimeError('El reporte no es un resultado completo de versión 2.0')
    path=manifest.parent/'resultados.parquet'
    before=(path.stat().st_size,path.stat().st_mtime_ns)
    def lit(v): return "'"+str(v).replace("'","''")+"'"
    def emit(name,query):
        cur=con.execute(query)
        cols=[r[0] for r in cur.description]
        rows=[dict(zip(cols,r)) for r in cur.fetchall()]
        tablas[name] = pd.DataFrame(rows, columns=cols)
    
    with tempfile.TemporaryDirectory(prefix='secop_diagnostico_') as temp:
        con=duckdb.connect()
        try:
            con.execute("SET memory_limit='1GB'")
            con.execute('CREATE MACRO tope_valor() AS '+str(float(tope)))
            con.execute('CREATE MACRO factor_valor() AS '+str(float(factor)))
            con.execute('SET threads=1')
            con.execute('SET preserve_insertion_order=false')
            con.execute('SET temp_directory='+lit(temp))
            print('Lectura de columnas necesarias del Parquet 2.0...',flush=True)
            con.execute(f'''CREATE TEMP TABLE base AS SELECT id_contrato,year(firma) anio,
                tipo_normalizado,valor_contrato valor,clasificacion,d1_concentracion d1,
                d2_valor_atipico d2,alerta_d3 d3,metodo_comparacion
                FROM read_parquet({lit(path.as_posix())})''')
            # Mediana de todos los valores finitos y positivos del tipo-año,
            # incluyendo los mayores al tope: aplicación literal independiente de a/b.
            con.execute('''CREATE TEMP TABLE medianas AS SELECT tipo_normalizado,anio,
                median(valor) mediana,count(*) n_referencia,
                median(valor) FILTER(WHERE valor<=tope_valor()) mediana_sin_tope
                FROM base WHERE isfinite(valor) AND valor>0
                    AND tipo_normalizado IS NOT NULL AND anio IS NOT NULL GROUP BY ALL''')
            con.execute('''CREATE TEMP VIEW flags AS SELECT b.*,m.mediana,m.n_referencia,
                valor/mediana ratio,
                (valor IS NULL OR NOT isfinite(valor) OR valor<=0) c,
                coalesce(isfinite(valor) AND valor>tope_valor(),false) a,
                coalesce(isfinite(valor) AND valor>0 AND valor>factor_valor()*mediana,false) b,
                coalesce(isfinite(valor) AND valor>0 AND valor>factor_valor()*mediana_sin_tope,false) b_sin_tope
                FROM base b LEFT JOIN medianas m
                ON b.tipo_normalizado=m.tipo_normalizado AND b.anio=m.anio''')
            con.execute('''CREATE TEMP VIEW diagnostic AS SELECT *,
                CASE WHEN c THEN 'C_no_positivo_no_finito_o_nulo'
                     WHEN a AND b THEN 'A_y_B_tope_y_mediana'
                     WHEN a THEN 'A_solo_tope'
                     WHEN b THEN 'B_solo_mediana' ELSE 'Sin_marca' END motivo
                FROM flags''')
            emit('overview','''SELECT count(*) total,count(*) FILTER(WHERE a OR b OR c) invalidos,
                count(*) FILTER(WHERE a OR b) extremos_nuevos,
                count(*) FILTER(WHERE clasificacion='ALERTA') alertas_actuales,
                count(*) FILTER(WHERE d2=1) d2_actuales,
                count(*) FILTER(WHERE (a OR b OR c) AND clasificacion='ALERTA') invalidos_alerta,
                count(*) FILTER(WHERE (a OR b OR c) AND d2=1) invalidos_d2,
                count(*) FILTER(WHERE (a OR b OR c) AND clasificacion='ALERTA' AND d2=1) invalidos_alerta_y_d2,
                sum(valor) FILTER(WHERE NOT c) suma_actual,
                sum(valor) FILTER(WHERE NOT c AND clasificacion='ALERTA') suma_alerta_actual,
                sum(valor) FILTER(WHERE NOT (a OR b OR c)) suma_pasa_reglas,
                sum(valor) FILTER(WHERE NOT (a OR b OR c) AND clasificacion='ALERTA') suma_alertas_que_pasan,
                count(*) FILTER(WHERE NOT c AND mediana IS NULL) sin_grupo_mediana,
                count(*) FILTER(WHERE b IS DISTINCT FROM b_sin_tope) cambios_mediana_excluir_tope
                FROM diagnostic''')
            metrics="count(*) n,count(*) FILTER(WHERE clasificacion='ALERTA') alerta,count(*) FILTER(WHERE d2=1) d2_1,count(*) FILTER(WHERE clasificacion='ALERTA' AND d2=1) alerta_y_d2"
            emit('exclusive_reasons',f'SELECT motivo,{metrics} FROM diagnostic GROUP BY motivo ORDER BY motivo')
            emit('year_reason',f"SELECT anio,motivo,{metrics} FROM diagnostic WHERE motivo<>'Sin_marca' GROUP BY anio,motivo ORDER BY anio,motivo")
            emit('year_total',f"SELECT anio,{metrics},sum(valor) FILTER(WHERE NOT c) suma_actual,sum(valor) FILTER(WHERE NOT(a OR b OR c)) suma_pasa FROM diagnostic GROUP BY anio ORDER BY anio")
            emit('top20','''SELECT id_contrato,anio,tipo_normalizado,valor,clasificacion,d1,d2,d3,
                mediana,ratio,motivo FROM diagnostic WHERE isfinite(valor)
                ORDER BY valor DESC,id_contrato LIMIT 20''')
            emit('relative_only_sample','''SELECT id_contrato,anio,tipo_normalizado,valor,mediana,ratio,
                n_referencia,clasificacion,d1,d2,d3 FROM diagnostic
                WHERE b AND NOT a ORDER BY valor DESC,id_contrato LIMIT 10''')
            emit('basic_reason','''SELECT CASE WHEN valor IS NULL THEN 'nulo' WHEN NOT isfinite(valor) THEN 'no_finito'
                WHEN valor=0 THEN 'cero' ELSE 'negativo' END motivo,count(*) n FROM diagnostic WHERE c GROUP BY 1''')
            emit('mediana_small','''SELECT min(n_referencia) minimo_grupo_marcado,
                count(*) FILTER(WHERE n_referencia<30) marcados_grupo_menos30
                FROM diagnostic WHERE b''')
            emit('small_group_case','''SELECT id_contrato,anio,tipo_normalizado,valor,mediana,n_referencia,ratio
                FROM diagnostic WHERE b AND n_referencia<30''')
            emit('method','''SELECT metodo_comparacion,count(*) n,count(*) FILTER(WHERE d2=1) alertas_d2
                FROM diagnostic GROUP BY metodo_comparacion''')
            # Para documentar los casos señalados sin tomar los nombres por claves.
            ids=','.join(lit(r[0]) for r in con.execute('SELECT id_contrato FROM diagnostic WHERE b AND NOT a ORDER BY valor DESC,id_contrato LIMIT 5').fetchall())
            emit('relative_context',f'''SELECT id_contrato,nombre_entidad,proveedor_adjudicado,
                substr(objeto_del_contrato,1,350) objeto FROM read_parquet({lit(path.as_posix())})
                WHERE id_contrato IN ({ids})''')
        finally:
            con.close()
    after=(path.stat().st_size,path.stat().st_mtime_ns)
    if before != after:
        raise RuntimeError('El Parquet cambió durante la lectura; repita el diagnóstico con una ejecución terminada.')
    tablas['metadatos'] = pd.DataFrame([{'archivo': str(path), 'version': report['parametros']['version'], 'corte': report['parametros']['cutoff'], 'tope': tope, 'factor': factor}])
    return tablas


In [ ]:
diagnostico = diagnosticar_valores(ROOT, LOTE, TOPE_ABSOLUTO, FACTOR_MEDIANA)
display(diagnostico['metadatos'])


## 2. Cuántos contratos se marcan y cuáles están en ALERTA

Estas tablas describen la clasificación **anterior a cualquier recálculo**. Los valores marcados se conservan en el universo; el diagnóstico no elimina contratos.


In [ ]:
display(diagnostico['overview'].T.rename(columns={0:'Resultado'}))
display(diagnostico['exclusive_reasons'].rename(columns={
    'motivo':'Motivo','n':'Contratos','alerta':'Hoy ALERTA','d2_1':'Hoy D2=1','alerta_y_d2':'Hoy ALERTA y D2=1'}))
display(diagnostico['basic_reason'])
r = diagnostico['overview'].iloc[0]
print(f"Valores positivos extremos nuevos: {int(r.extremos_nuevos):,}.")
print(f"Marcados en ALERTA: {int(r.invalidos_alerta):,} de {int(r.alertas_actuales):,} alertas actuales.")
print(f"Positivos sin grupo de mediana: {int(r.sin_grupo_mediana):,}.")
print(f"Cambios al excluir primero el tope de la mediana: {int(r.cambios_mediana_excluir_tope):,}.")


## 3. Motivos por año

Permite distinguir si la anomalía se concentra en ciertos años y si afecta a las alertas o a contratos todavía no evaluables. Las combinaciones ausentes en la tabla corresponden a conteos cero. El último año puede tener cobertura parcial según el corte del reporte.


In [ ]:
display(diagnostico['year_reason'].rename(columns={
    'anio':'Año','motivo':'Motivo','n':'Contratos','alerta':'Hoy ALERTA','d2_1':'Hoy D2=1','alerta_y_d2':'Hoy ALERTA y D2=1'}))


## 4. Los 20 contratos de mayor valor

Se muestran valores originales en COP y una columna en billones (10¹² COP), junto a la mediana, razón valor/mediana, motivo y señales actuales. Un NULL en una dimensión significa **no evaluable**, no cero.


In [ ]:
mayores = diagnostico['top20'].copy()
mayores['valor_billones_COP'] = mayores.valor/1e12
display(mayores.style.format({'valor':'{:,.2f}','valor_billones_COP':'{:,.6f}',
                             'mediana':'{:,.2f}','ratio':'{:,.2f}'}, na_rep='No evaluable'))


## 5. Riesgo de excluir operaciones legítimas

Revisar únicamente los mayores valores no basta: la regla relativa puede señalar convenios de infraestructura o administración de fondos cuyos montos sean reales. Las siguientes tablas muestran casos **solo B**, su contexto y el tamaño de la referencia. Los objetos contractuales no prueban legitimidad ni exactitud del monto; orientan la revisión documental.

Un grupo pequeño genera una mediana menos estable. No se impone un mínimo nuevo en este diagnóstico: se muestra cuántas marcas proceden de grupos menores a 30.


In [ ]:
display(diagnostico['relative_only_sample'])
with pd.option_context('display.max_colwidth', 350):
    display(diagnostico['relative_context'])
display(diagnostico['mediana_small'])
display(diagnostico['small_group_case'])


## 6. Comparación monetaria: filtro actual frente a reglas propuestas

**Esta comparación solo filtra montos. No es un antes/después del modelo.** Las clasificaciones siguen siendo las de 2.0. Los contratos que pasan estas reglas no quedan acreditados como correctos; solo dejan de ser señalados por estos criterios.


In [ ]:
r = diagnostico['overview'].iloc[0]
comparacion_montos = pd.DataFrame([
    {'escenario':'Filtro actual: finito y positivo', 'monto_COP':r.suma_actual},
    {'escenario':'Pasa A/B/C propuestas', 'monto_COP':r.suma_pasa_reglas},
    {'escenario':'ALERTA actual, filtro monetario actual', 'monto_COP':r.suma_alerta_actual},
    {'escenario':'ALERTA actual, solo montos que pasan A/B/C', 'monto_COP':r.suma_alertas_que_pasan},
])
comparacion_montos['billones_COP'] = comparacion_montos.monto_COP/1e12
display(comparacion_montos.style.format({'monto_COP':'{:,.2f}','billones_COP':'{:,.6f}'}))
anual = diagnostico['year_total'].copy()
anual['billones_actuales'] = anual.suma_actual/1e12
anual['billones_pasan_reglas'] = anual.suma_pasa/1e12
display(anual[['anio','n','billones_actuales','billones_pasan_reglas']])
if r.suma_actual > 0:
    print(f"Los valores excluidos del monto representan {100*(1-r.suma_pasa_reglas/r.suma_actual):.4f}% del monto actual.")
    print(f"Las alertas actuales concentran {100*r.suma_alerta_actual/r.suma_actual:.2f}% del monto actual.")
print('La clasificación nueva requiere recalcular referencias y señales: no se obtiene restando las alertas marcadas.')


## 7. Cómo calcula el pipeline las señales

**D2:** logaritmo natural del valor, comparables por UNSPSC, tipo y causal en los 24 meses previos, excluyendo el día de firma. Con al menos 30 comparables, calcula mediana y MAD. Señala `(ln(valor) − mediana)/(1,4826 × MAD) > 3,5`. Si MAD=0, compara con P95. Primero intenta UNSPSC8 y luego UNSPSC4. No utiliza media ni desviación estándar convencional.

**D1:** participación del valor del proveedor en el valor de la entidad durante 365 días, incluyendo el contrato y su día. El umbral actual es 40%, con mínimos de contratos, proveedores y cobertura. Por ello, valores extremos pueden alterar tanto la señal del proveedor como las participaciones de otros proveedores.

**D3:** acumulación de contratos de diciembre por entidad-año, retrospectiva. No depende directamente del monto.

La tabla siguiente muestra cuántos contratos evaluó cada método D2 en el resultado existente.


In [ ]:
display(diagnostico['method'])


## 8. Decisiones pendientes antes de una versión 2.1

1. Confirmar las reglas de exclusión y conservar el valor original, junto al motivo y mediana de referencia. «No utilizable según regla» no equivale a «digitación errónea confirmada».
2. Para D1, decidir si la presencia de un valor inválido mantiene toda la ventana como no evaluable (comportamiento estricto actual) o si se calcula con montos aceptados y se documenta la cobertura. Los contratos con valor excluido deben quedar no evaluables en las señales monetarias.
3. Reconocer que la mediana del año completo introduce información posterior a la firma: la validación será retrospectiva al corte.
4. Crear un lote independiente y versión 2.1: cambiar solo la etiqueta de versión no garantiza preservar los resultados 2.0.
5. Recalcular D1 y D2, mantener la regla de 2 de 3 y comparar resultados reales por ID, sin sobrescribir el lote anterior.

**Este notebook termina en diagnóstico. No implementa ni ejecuta esos cambios.**
